# aiayn: Kaggle training → evaluation → comparison → preserved outputs

One unattended session trains every run in `RUNS`, in order (docs/KAGGLE.md §8–10,
ADR-013, ADR-020, ADR-021):

1. **Auto-resume:** a run whose name exists in an attached input (`/kaggle/input/**`:
   a previous notebook output or the `aiayn-runs` dataset) continues exactly; a finished
   run goes straight to evaluation.
2. Each run gets the time left in the session (minus a reserve for evaluation and
   publishing); a run that does not finish stops cleanly and resumes next session.
3. Completed runs: test-set evaluation (BLEU, chrF) and `REPORT.md`.
4. `runs/COMPARISON.md`: every run against the first one (the Transformer reference),
   with fairness checks and paired bootstrap significance.
5. **Publish** all runs to the private dataset `aiayn-runs`.

**Always launch with Save Version → Save & Run All (Commit).** Interactive sessions lose
`/kaggle/working` when they end. Settings: Accelerator **GPU T4**, Internet **ON**.
To publish to `aiayn-runs`, add the secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`
(*Add-ons → Secrets*). Without them the notebook output is the only copy.

In [ ]:
# 0. Parameters
GITHUB_URL = "https://github.com/mohamed242006/Transformer-from-scratch-with-a-paper-reproduction.git"
GITHUB_BRANCH = "master"
RUNS = [  # (experiment config, stable run name); the first run is the comparison reference
    ("configs/experiment/multi30k_tiny.yaml", "multi30k_tiny_s1"),
    ("configs/experiment/multi30k_bilstm_luong.yaml", "multi30k_bilstm_luong_s1"),
    ("configs/experiment/multi30k_gru_bahdanau.yaml", "multi30k_gru_bahdanau_s1"),
    ("configs/experiment/multi30k_lstm_seq2seq.yaml", "multi30k_lstm_seq2seq_s1"),
]
OVERRIDES = []  # new runs only, e.g. ["train.seed=2"] (then use *_s2 run names)
AUTO_RESUME = True
EVALUATE = True  # completed runs: best beam 4, averaged beam 4, best greedy
COMPARE = True  # runs/COMPARISON.md on test.averaged.beam4
PUBLISH = True  # upload every run to the private dataset below
RUNS_DATASET = "aiayn-runs"
SESSION_MINUTES = 690  # 11.5 h of Kaggle's 12 h (ADR-013 §5)
RESERVE_MINUTES = 25  # kept back for evaluation, comparison and publishing
WORK = "/kaggle/working"

In [ ]:
# 1. Code and dependencies (torch is never reinstalled)
import glob
import importlib.util
import os
import shutil
import subprocess
import sys
import time

SESSION_START = time.time()
REPO = f"{WORK}/aiayn"
if not os.path.exists(f"{REPO}/pyproject.toml"):
    clone = ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_URL, REPO]
    subprocess.run(clone, check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
needed = {"sentencepiece": "sentencepiece>=0.2.0", "sacrebleu": "sacrebleu>=2.4"}
missing = [spec for module, spec in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacremoses==0.2.0"], check=True)  # pinned: decides IWSLT14 split membership (ADR-023)
sys.path.insert(0, f"{REPO}/src")  # the editable install is only visible to new processes
head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code: git", head)

In [ ]:
# 2. Kaggle API credentials from notebook secrets (needed only for publishing)
KAGGLE_USER = None
try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER = secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
except Exception as exc:  # no secrets configured: the commit output stays the only copy
    print("publishing disabled (no KAGGLE_USERNAME/KAGGLE_KEY secrets):", type(exc).__name__)
print("publish to dataset:", bool(PUBLISH and KAGGLE_USER), f"({KAGGLE_USER}/{RUNS_DATASET})")

In [ ]:
# 3. Train every run (auto-resume). Exit code 0 = completed, 3 = stopped on the budget.
from aiayn.diagnostics.kaggle_runs import latest_run, run_progress


def minutes_left():
    return SESSION_MINUTES - (time.time() - SESSION_START) / 60


def train(experiment, name):
    run_dir = f"{WORK}/runs/{name}"
    previous = None
    if AUTO_RESUME and not os.path.exists(run_dir):
        previous = latest_run(name, ["/kaggle/input"], unzip_into=f"{WORK}/_restored")
    budget = int(minutes_left() - RESERVE_MINUTES)
    if budget < 5:
        print(f"[{name}] skipped: session budget used up")
        if previous:  # keep the restored copy so it is published unchanged
            shutil.copytree(previous, run_dir, dirs_exist_ok=True)
        return run_dir, 3
    cmd = [sys.executable, "-u", "scripts/train.py", "--run-dir", run_dir, "--prepare-data"]
    if previous:
        print(f"[{name}] resuming from {previous} (update {run_progress(previous)})")
        cmd += ["--resume", str(previous)]
    elif os.path.exists(f"{run_dir}/checkpoints"):
        cmd += ["--resume", run_dir]
    else:
        print(f"[{name}] new run")
        cmd += ["--experiment", experiment]
        for item in OVERRIDES:
            cmd += ["--override", item]
    cmd += ["--override", f"train.time_budget_minutes={budget}"]
    print("$", " ".join(cmd))
    code = subprocess.run(cmd).returncode
    print(f"[{name}] training exit code {code}")
    return run_dir, code


RESULTS = {}
for experiment, name in RUNS:
    RESULTS[name] = train(experiment, name)
    assert RESULTS[name][1] in (0, 3), f"{name}: training failed, see the log above"

In [ ]:
# 4. Evaluate completed runs on the test set (skips existing results), then REPORT.md
for name, (run_dir, code) in RESULTS.items():
    if code != 0:
        print(f"[{name}] not completed: run this notebook again to resume it")
        continue
    if EVALUATE:
        for checkpoint, extra in [("best", []), ("averaged", []), ("best", ["--greedy"])]:
            tag = f"test.{checkpoint}.{'greedy' if extra else 'beam4'}.json"
            if os.path.exists(f"{run_dir}/eval/{tag}"):
                print(f"[{name}] exists: {tag}")
                continue
            evaluate = [sys.executable, "scripts/evaluate.py", "--run-dir", run_dir]
            evaluate += ["--split", "test", "--checkpoint", checkpoint, *extra]
            subprocess.run(evaluate, check=True)
    subprocess.run([sys.executable, "scripts/report.py", "--run-dir", run_dir], check=True)

In [ ]:
# 5. Fair comparison against the first run (fairness checks + paired bootstrap)
TAG = "test.averaged.beam4"
ready = [d for d, _ in RESULTS.values() if os.path.exists(f"{d}/eval/{TAG}.json")]
reference_dir = RESULTS[RUNS[0][1]][0]
if COMPARE and len(ready) >= 2 and reference_dir in ready:
    out = f"{WORK}/runs/COMPARISON.md"
    cmd = [sys.executable, "scripts/compare_runs.py", "--tag", TAG, "--out", out]
    for run_dir in ready:
        cmd += ["--run", run_dir]
    subprocess.run(cmd)  # exit code 1 = a fairness check failed (shown in the file)
    shutil.copy(out, f"{reference_dir}/COMPARISON.md")  # published with the reference run
else:
    print("comparison skipped: needs the reference and one more evaluated run")

In [ ]:
# 6. Publish every run to the private dataset so it survives this session
existing = [d for d, _ in RESULTS.values() if os.path.exists(f"{d}/metadata.json")]
if PUBLISH and KAGGLE_USER and existing:
    attached = glob.glob(f"/kaggle/input/{RUNS_DATASET}")
    cmd = [sys.executable, "scripts/kaggle_publish_runs.py"]
    for run_dir in existing:
        cmd += ["--run", run_dir]
    cmd += ["--user", KAGGLE_USER, "--slug", RUNS_DATASET]
    status = ", ".join(f"{n}: exit {c}" for n, (_, c) in RESULTS.items())
    cmd += ["--message", status[:250]]
    if attached:
        cmd += ["--previous", attached[0]]  # keep the other runs already in the dataset
    result = subprocess.run(cmd)
    print("published" if result.returncode == 0 else f"PUBLISH FAILED ({result.returncode})")
else:
    print("not published: the committed notebook output is the copy of these runs")
print("next session: attach this notebook's output or the", RUNS_DATASET, "dataset, Run All")